# v8.2 RQ2 EMA-aware Feature-Family Diagnostic

## tl;dr

The focused RQ2 revision is complete. Under an identical regularised Logistic
Regression framework, the six-feature EMA-aware Technical family achieved
**69.60%** mean development OOF
Accuracy and **63.88%** retrospective audit
Accuracy (Balanced Accuracy **63.77%**).

It exceeded the weekly-trend persistence baseline by
**7.62 pp**, with a 20-day paired
moving-block bootstrap 95% interval of
**[0.74 pp, 13.51 pp]**.

The EMA state itself provided only a small increment over Core-5:
**0.24 pp** in development OOF and
**0.74 pp** in the later audit. Both
intervals include or touch zero. Therefore EMA8/13 is retained because it gives
a theoretically interpretable bridge to RQ4 and slightly improves fold
stability, not because it has been shown to produce a large or statistically
secure Accuracy gain.

Across the five every-fifth-date phase checks, Technical achieved the highest
family Accuracy in **4/5** phases. Its phase Accuracy
ranged from **60.49%** to
**74.07%**, so the aggregate 63.88%
should not be presented as uniformly stable over time.


## Context & Methods

RQ2 asks:

> Under an identical regularised Logistic Regression framework, how does
> retrospective walk-forward classification performance differ across four
> pre-specified standalone feature families—EMA-aware technical, cross-market,
> macroeconomic, and broad sentiment/risk proxies—when predicting Tesla's
> five-session forward-versus-trailing mean-close direction?

The target is

$$
y_t=\mathbb{1}\left[
\frac{1}{5}\sum_{i=1}^{5}Close_{t+i}
>
\frac{1}{5}\sum_{i=0}^{4}Close_{t-i}
\right].
$$

The signal information cutoff is after `Close[t]`. Development uses five
chronological folds with 250 test rows and a five-row purge. The common model
uses training-fold median imputation, training-fold standardisation,
`C=0.5`, balanced class weights, and a fixed 0.50 threshold.

### Key assumptions and boundaries

- `EMA8_13_State = 1[EMA8[t] > EMA13[t]]` is a persistent state, not a
  one-day crossover event.
- `SMA10_20_State` is a sensitivity design; raw moving-average price levels are
  not stacked into the main family.
- The 2024–2026 period is a retrospective temporal audit because the historical
  results had already been inspected during redesign.
- The target is a weekly mean-price classification task, not a trading win rate.
- RQ3 model comparisons and RQ4 economic backtests are intentionally excluded.


## Data


In [ ]:
from pathlib import Path
import pandas as pd

try:
    from IPython.display import display, Image
except ImportError:
    def display(value):
        print(value)
    def Image(filename):
        return f'[Image: {filename}]'

working_directory = Path.cwd().resolve()
if (working_directory / 'outputs').exists():
    ROOT = working_directory
elif (working_directory.parent / 'outputs').exists():
    ROOT = working_directory.parent
else:
    raise FileNotFoundError(
        'Run from the v8.2 project root or its notebooks directory.'
    )
OUTPUTS = ROOT / 'outputs'
FIGURES = ROOT / 'figures'

family = pd.read_csv(OUTPUTS / 'rq2_feature_family_summary.csv')
technical = pd.read_csv(OUTPUTS / 'rq2_technical_design_summary.csv')
family_bootstrap = pd.read_csv(
    OUTPUTS / 'rq2_feature_family_paired_block_bootstrap.csv'
)
technical_bootstrap = pd.read_csv(
    OUTPUTS / 'rq2_technical_design_paired_block_bootstrap.csv'
)
family_phase = pd.read_csv(
    OUTPUTS / 'rq2_feature_family_phase_offset_metrics.csv'
)
validation = pd.read_csv(OUTPUTS / 'rq2_validation_checks.csv')


## Results


### 1. Standalone feature-family comparison

| Design | Development_Mean_Accuracy | Development_Accuracy_SD | Audit_Accuracy | Audit_Balanced_Accuracy | Audit_Macro_F1 |
|---|---:|---:|---:|---:|---:|
| Technical — EMA-aware compact | 69.60% | 2.62 pp | 63.88% | 63.77% | 0.6378 |
| Broad sentiment and risk proxies | 55.44% | 5.42 pp | 58.97% | 58.22% | 0.5749 |
| Cross-market | 64.96% | 4.64 pp | 57.99% | 58.34% | 0.5788 |
| Macroeconomic | 49.84% | 7.70 pp | 50.12% | 50.83% | 0.4930 |

Technical produced the highest point estimate in both development and the
later retrospective audit. Macroeconomic features alone remained close to
chance. Cross-market and broad sentiment/risk proxies showed moderate
information, but their later ordering differed from the development ordering.


In [ ]:
display(
    family[
        [
            'Feature Family',
            'Development_Mean_Accuracy',
            'Development_Accuracy_SD',
            'Audit_Accuracy',
            'Audit_Balanced_Accuracy',
            'Audit_Macro_F1',
            'Audit_ROC_AUC',
        ]
    ].sort_values('Audit_Accuracy', ascending=False)
)
display(Image(filename=str(FIGURES / 'rq2_feature_family_audit.png')))


### 2. Pairwise uncertainty

Technical exceeded Macroeconomic in the audit with an interval above zero. Its
comparison with Broad sentiment remained statistically unresolved. The lower
bound against Cross-market touched exactly zero, so that contrast should be
treated as borderline rather than conclusive.


In [ ]:
uncertainty = family_bootstrap.copy()
for column in ['Point Difference', 'CI Lower 95%', 'CI Upper 95%']:
    uncertainty[column] = uncertainty[column] * 100
display(uncertainty)


### 3. Moving-average expression sensitivity

| Design | Development_Mean_Accuracy | Development_Accuracy_SD | Audit_Accuracy | Audit_Balanced_Accuracy | Audit_Macro_F1 |
|---|---:|---:|---:|---:|---:|
| Legacy SMA50-ratio comparator | 69.68% | 3.56 pp | 63.88% | 63.75% | 0.6376 |
| Primary EMA-aware compact | 69.60% | 2.62 pp | 63.88% | 63.77% | 0.6378 |
| SMA10/20-state sensitivity | 69.44% | 2.88 pp | 63.39% | 63.32% | 0.6332 |
| Core-5 without moving-average state | 69.36% | 2.84 pp | 63.14% | 63.04% | 0.6305 |

The EMA-aware design had the smallest fold standard deviation, but its
development Accuracy was 0.08 percentage points below the legacy SMA50-ratio
comparator. The EMA and legacy designs both reached 63.88% audit Accuracy.
This supports selecting EMA for conceptual alignment and stability, not for a
claim of dominant predictive power. In the stricter phase-specific refit,
the SMA sensitivity pooled to **64.86%**, compared with
**63.88%** for EMA. This further confirms that EMA is not
uniformly superior across specifications.


In [ ]:
display(technical)
display(technical_bootstrap)
display(Image(filename=str(FIGURES / 'rq2_technical_design_comparison.png')))


### 4. Five phase-offset robustness checks

Each phase retains every fifth eligible date and refits its own Logistic
Regression using phase-specific development rows. Future five-day windows do
not overlap within a phase, although adjacent phase labels still share a
comparison window and therefore are not fully independent trials.


In [ ]:
phase_accuracy = family_phase.pivot(
    index='Phase Offset',
    columns='Feature Family',
    values='Accuracy',
)
display(phase_accuracy)


## Takeaways


1. Use the EMA-aware six-feature Technical family as the RQ2 main
   specification.
2. Report SMA10/20, Core-5, and legacy SMA50 as sensitivity or ablation
   evidence.
3. State that EMA8/13 improved interpretability and reduced fold variability,
   while its incremental Accuracy contribution was modest and statistically
   unresolved.
4. Carry the same `EMA8 > EMA13` definition into RQ4 as the transparent rule
   comparator. RQ4—not RQ2—will test whether the state can support economic
   returns.
5. Preserve the retrospective claim boundary until a genuinely new future
   period is observed.


In [ ]:
display(validation)
assert validation['Status'].eq('Pass').all()
